In [ ]:
# Extract idioms from detect_result.json
import json
import re
from pathlib import Path


def extract_outputs(text):
    """Extract phrases after each '***Output: ## YES ##' block."""
    if not isinstance(text, str):
        return []

    output_blocks = re.findall(
        r"\*\*\*Output:\s*##\s*YES\s*##\s*(.*?)(?=assistant|\n\s*\n|$)",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    if not output_blocks:
        return []

    items = []
    for raw in output_blocks:
        raw = re.sub(r"assistant\s*$", "", raw.strip(), flags=re.IGNORECASE)
        if not raw:
            continue

        numbered_items = re.findall(r"\d+\.\s*(.*?)(?=\s*\d+\.|$)", raw)
        if numbered_items:
            items.extend(numbered_items)
        else:
            items.extend(re.split(r"[,;\n]+", raw))

    result = []
    seen = set()
    for item in items:
        cleaned = item.strip(" .-\t\r\n'\"")
        key = cleaned.lower()
        if cleaned and key not in seen:
            result.append(cleaned)
            seen.add(key)
    return result


def find_input_file():
    candidates = [
        Path("detect_result.json"),
        Path("eval") / "detect" / "result" / "detect_result.json",
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        "Cannot find detect_result.json. Run this notebook from the project root "
        "or from eval/detect/result."
    )


input_file_path = find_input_file()
output_file_path = input_file_path.with_name("detect_idiom.json")

with input_file_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

for entry in data:
    entry["detect_idiom"] = extract_outputs(entry.get("dpo_eval_8B", ""))

with output_file_path.open("w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Read {len(data)} records from {input_file_path}")
print(f"Wrote {output_file_path}")
